# Potential Talents – Exploratory Data Analysis (EDA)

## Objective

The purpose of this notebook is to understand the structure, signal strength, and domain characteristics of the candidate dataset in order to design a robust ranking system.

Key goals:

- Understand the distribution of HR-related vs non-HR candidates
- Identify meaningful textual signals (seniority, intent, domain relevance)
- Detect duplicate concentration and vocabulary patterns
- Determine whether supervised modeling is appropriate

This notebook focuses purely on data understanding and feature discovery.

In [53]:
import re
import numpy as np
import pandas as pd
from collections import Counter

In [54]:
# data load
KEYWORDS = "aspiring human resources seeking human resources"

csv_path = "../data/potential-talents.csv"

df = pd.read_csv(csv_path)
df.head()


,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


### Helper Functions

#### Text Normalization Strategy

To ensure consistent textual analysis:

- Convert to lowercase
- Remove excessive whitespace
- Standardize punctuation handling

In [55]:
def norm_text(s):
    if pd.isna(s): 
        return ""
    s = str(s).lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s\+\-\/]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [56]:
def parse_connections(x):
    if pd.isna(x): return None
    x = str(x).strip().lower()
    if x.endswith("+"):
        x = x[:-1]
    x = re.sub(r"[^\d]", "", x)
    return int(x) if x else None

In [57]:
def bucket_conn(v):
    if v is None: return "missing"
    if v <= 50: return "0-50"
    if v <= 200: return "51-200"
    if v <= 500: return "201-500"
    return "500+"

### Dataset Overview

The dataset contains anonymized candidate profiles with the following attributes:

- `id` – unique candidate identifier
- `job_title` – textual description of candidate’s role
- `location` – geographical information
- `connection` – number of professional connections (categorical text)

Notably:

- The dataset contains 104 rows.
- No missing values exist.
- No labeled `fit` values are provided.

Because the target variable (`fit`) is unavailable, this problem must be approached as a ranking problem rather than supervised regression/classification.

In [58]:
# Basic checks
print("Rows:", len(df))
print("Unique ids:", df["id"].nunique())
print("\nMissingness:")
print(df[["job_title","location","connection"]].isna().sum())

# Normalize columns
df["job_title_n"] = df["job_title"].apply(norm_text)
df["location_n"] = df["location"].apply(norm_text)
df["conn_n"] = df["connection"].apply(parse_connections)
df["conn_bucket"] = df["conn_n"].apply(bucket_conn)

print("\nTop raw job titles:")
print(df["job_title"].value_counts().head(15))

print("\nTop normalized job titles:")
print(df["job_title_n"].value_counts().head(15))

print("\nConnections bucket distribution:")
print(df["conn_bucket"].value_counts(dropna=False))

# Keyword coverage
kw_terms = [t for t in norm_text(KEYWORDS).split() if t not in {"and","or","the","a","an"}]
def kw_hits(text):
    return sum(1 for t in kw_terms if t in text)

df["kw_hits"] = df["job_title_n"].apply(kw_hits)

print("\nKeyword term list:", kw_terms)
print("\nCandidates with >=1 keyword hit in job_title:", (df["kw_hits"] >= 1).sum())
print("Candidates with >=2 keyword hits in job_title:", (df["kw_hits"] >= 2).sum())

print("\nExamples with highest keyword hits:")
print(df.sort_values("kw_hits", ascending=False)[["id","job_title","location","connection","kw_hits"]].head(15).to_string(index=False))

Rows: 104
Unique ids: 104

Missingness:
job_title     0
location      0
connection    0
dtype: int64

Top raw job titles:
job_title
2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional       7
Aspiring Human Resources Professional                                                                          7
Student at Humber College and Aspiring Human Resources Generalist                                              7
People Development Coordinator at Ryan                                                                         6
Native English Teacher at EPIK (English Program in Korea)                                                      5
Aspiring Human Resources Specialist                                                                            5
HR Senior Specialist                                                                                           5
Advisory Board Member at Celal Bayar University                              

### Strategic Observation
* Because so many candidates explicitly say “Aspiring Human Resources”:
* A naive TF-IDF model will rank them all nearly equally.
* It will struggle to meaningfully separate them.

### We must differentiate:
* “aspiring HR student” vs
* “HR Senior Specialist” vs
* “SVP, CHRO”

Seniority + relevance ranking problem, not just keyword matching.

### Token Frequency Analysis

We examine the most common tokens in job titles to understand domain composition.

Key observations:

- "human" and "resources" dominate the vocabulary.
- "aspiring" appears frequently.
- Seniority indicators such as "senior", "manager", "director" are less frequent.
- Non-HR domain terms (e.g., "teacher", "biology", "engineer") appear in a minority subset.

This indicates the dataset is already partially filtered toward HR-related candidates.

In [59]:
def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

all_tokens = []

for t in df["job_title_n"]:
    all_tokens.extend(tokenize(t))

token_counts = Counter(all_tokens)

print("\nTop 40 Most Common Tokens:")
for word, count in token_counts.most_common(40):
    print(word, count)


Top 40 Most Common Tokens:
human 63
resources 63
at 46
aspiring 35
and 28
professional 20
student 16
seeking 15
college 14
generalist 14
university 12
specialist 12
of 11
business 11
english 10
in 10
coordinator 10
c 7
t 7
bauer 7
graduate 7
magna 7
cum 7
laude 7
humber 7
management 7
manager 7
people 6
development 6
ryan 6
hr 6
senior 6
native 5
teacher 5
epik 5
program 5
korea 5
the 5
an 5
advisory 4


### Domain Classification Heuristics

To understand dataset structure, we introduce heuristic flags:

- HR Domain Indicator
- Seniority Indicator
- Entry / Intent Indicator

These are simple rule-based flags used for exploratory purposes.

In [60]:
def classify_title(text):
    text = text.lower()

    is_hr = any(term in text for term in [
        "human resources", " hr ", "generalist",
        "specialist", "coordinator", "chro"
    ])

    is_senior = any(term in text for term in [
        "senior", "manager", "director", "vp", "chro"
    ])

    is_student = any(term in text for term in [
        "student", "aspiring", "seeking", "graduate"
    ])

    return is_hr, is_senior, is_student

df["hr_flag"], df["senior_flag"], df["student_flag"] = zip(
    *df["job_title_n"].apply(classify_title)
)

print("HR domain count:", df["hr_flag"].sum())
print("Senior count:", df["senior_flag"].sum())
print("Student/Entry count:", df["student_flag"].sum())

HR domain count: 77
Senior count: 20
Student/Entry count: 54


In [61]:
non_hr = df[~df["hr_flag"]]

print("\nNon-HR Titles:")
print(non_hr["job_title"].unique())
print("\nCount:", len(non_hr))


Non-HR Titles:
<StringArray>
[                                                 'Native English Teacher at EPIK (English Program in Korea)',
                                                            'Advisory Board Member at Celal Bayar University',
                                                                              'Student at Chapman University',
                                                                   'Junior MES Engineer| Information Systems',
                                                                  'HR Manager at Endemol Shine North America',
                                         'RRP Brand Portfolio Executive at JTI (Japan Tobacco International)',
                                      'Bachelor of Science in Biology from Victoria University of Wellington',
                                                         'Undergraduate Research Assistant at Styczynski Lab',
                                                               'Lead Official at W

### Findings

- 77 candidates fall within HR-related domain.
- 20 candidates show seniority indicators.
- 54 candidates show entry-level or intent indicators.
- ~27 candidates are outside HR domain.

This confirms:

The dataset is HR-heavy but not purely HR-exclusive.
Ranking logic must handle both HR and non-HR candidates gracefully.

##### Duplicate Title Concentration

Several job titles appear multiple times (e.g., "Aspiring Human Resources Specialist").

This introduces a ranking bias risk:
* If scoring is purely keyword-based, duplicate titles may dominate the top results.
* Therefore, diversity control must be considered during model design.

##### Design Implications from EDA

1. The dataset is small (104 rows).
2. No labeled `fit` variable exists.
3. HR-related vocabulary dominates.
4. Many candidates share identical titles.
5. Seniority and intent signals are separable.
6. Hard filtering may remove borderline relevant candidates.

Conclusion:

This problem is framed as a ranking problem with adaptive feedback, not as a supervised prediction problem.

### EDA Summary

- Ranking > Classification
- Need structured scoring beyond keyword frequency
- Duplicate handling required
- Adaptive mechanism required for starring functionality